# 🐍 Python en IA — Partie 2 · Segmenter les clients d'*Amora Zone*

**Le cas :** *Amora Zone* est une entreprise française de cosmétiques naturels : produits de soin et ingrédients pour cosmétiques faits maison, vendus dans ses boutiques et sur son site e-commerce.

La directrice marketing t'envoie l'export des **commandes des 2 dernières années** et te pose une question :

> « Tous nos clients reçoivent la même newsletter. Peut-on les regrouper en quelques profils, pour envoyer à chacun le bon message ? »

### 🗺️ La feuille de route
On avance **une étape à la fois**. Chaque étape utilise le résultat de la précédente.

| Étape | Module | Ce qu'on fait | Ce qu'on obtient |
|---|---|---|---|
| 1 | M3 | Charger le fichier avec pandas | un tableau de commandes |
| 2 | M4 | Explorer les commandes | les premiers chiffres business |
| 3 | M5 | Passer des commandes aux clients | une ligne par client (le tableau RFM) |
| 4 | M6 | Lancer le k-means | un numéro de segment par client |
| 5 | M7 | Lire les segments | des profils nommés et un plan d'action |

### 🧭 Mode d'emploi
1. **Fichier → Enregistrer une copie dans Drive.**
2. Binôme : 🧑‍✈️ **Pilote** = tape le code · 🧭 **Copilote** = lit les consignes et guide. **On inverse à chaque module.**
3. Exécute les cellules **dans l'ordre**, avec **Maj + Entrée**. Souviens-toi de la Partie 1 : une cellule réutilise ce que les cellules exécutées avant elle ont créé.
4. 🔮 = prédis le résultat avant d'exécuter · `A_COMPLETER` = à compléter · 💡 = indice · 🚀 = pour aller plus loin · 🆘 = cellule de secours (la solution).
5. 🔁 Chaque module commence par une cellule **CHECKPOINT** qui recrée tout ce dont il a besoin. Si tu as une erreur, sauté une étape ou redémarré la session : exécute-la, puis continue.

# M3 · Étape 1 — Charger les commandes avec pandas
👥 **On inverse les rôles !**

### pandas, c'est quoi ?
`pandas` est une **librairie** : une boîte à outils écrite par d'autres, qu'on ajoute à Python. Elle sert à manipuler des **tableaux**, comme Excel, mais avec du code. Elle est déjà installée dans Colab ; il suffit de l'**importer**.

Deux mots de vocabulaire :
- un **DataFrame** = un tableau entier (des lignes et des colonnes) ;
- une **Series** = une seule colonne de ce tableau.

In [ ]:
import pandas as pd                  # on importe pandas et on lui donne le surnom court « pd »
import matplotlib.pyplot as plt      # la librairie des graphiques, surnommée « plt »

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "database.csv")

print("Fichier chargé ✅")

**Ce que fait chaque ligne :**
| Ligne | Ce qu'elle fait |
|---|---|
| `import pandas as pd` | ouvre la boîte à outils pandas ; ensuite, on écrit `pd.` pour utiliser un de ses outils |
| `URL_DATA = "https://..."` | une variable texte (M2.3) qui contient l'adresse du dossier de données sur GitHub |
| `pd.read_csv(...)` | lit un fichier **CSV** (un tableau en texte, colonnes séparées par des virgules) et le transforme en DataFrame |
| `URL_DATA + "database.csv"` | colle deux textes (M2.2) : l'adresse du dossier + le nom du fichier |
| `commandes = ...` | range le tableau dans la variable `commandes` |

### Regarder les premières lignes
`.head()` affiche les 5 premières lignes. C'est **toujours** la première chose à faire avec un nouveau fichier.

In [ ]:
commandes.head()

### 📖 Le dictionnaire des colonnes
**Une ligne = une commande**, passée à un instant précis. Un client qui rachète apparaît donc sur **plusieurs lignes**.

| Colonne | Contenu | Exemple |
|---|---|---|
| `id_commande` | identifiant unique de la commande | `CMD00001` |
| `id_client` | identifiant du client qui a commandé | `C0247` |
| `date_commande` | date et heure de la commande | `2024-11-01 10:37` |
| `canal` | où la commande a été passée | `boutique` ou `site web` |
| `ville` | ville de la boutique, ou ville de livraison pour le site | `Paris` |
| `latitude`, `longitude` | coordonnées GPS de la ville (pour la carte de la Partie 3) | `48.8566`, `2.3522` |
| `categorie` | famille de produits principale | `Soin visage`, `Ingrédients DIY`… |
| `nb_articles` | nombre d'articles dans le panier | `3` |
| `code_promo` | code utilisé (**vide** si aucun) | `SOLDES20` |
| `montant` | montant payé, remise déduite (€) | `88.00` |

👉 Dans `code_promo`, les cases vides s'affichent `NaN` (*Not a Number*). C'est la façon dont pandas écrit « **aucune valeur** » : ici, la commande n'a pas utilisé de code promo.

### La taille du tableau
🔮 **Combien de lignes et de colonnes ?** Prédis, puis exécute :

In [ ]:
commandes.shape

👉 `(4475, 11)` : **4 475 commandes** (lignes) et **11 colonnes**. `.shape` s'écrit sans parenthèses : ce n'est pas une action, c'est une information sur le tableau.

### Vérifier les types des colonnes
Premier réflexe d'un analyste, vu en M2.2 : **vérifier le type de chaque colonne**.

In [ ]:
commandes.dtypes

| `dtype` affiché | Type Python (M2.2) | Colonnes |
|---|---|---|
| `int64` | `int` | `nb_articles` |
| `float64` | `float` | `montant`, `latitude`, `longitude` |
| `object` ou `str` | `str` (texte) | `id_commande`, `id_client`, `canal`, `ville`, `categorie`, `code_promo`… **et `date_commande`** |

⚠️ **`date_commande` est lue comme du texte.** Pour Python, `"2024-11-01 10:37"` est une suite de caractères, comme un prénom. Impossible de calculer « combien de jours depuis cette date ». Il faut la **convertir en vraie date** avec `pd.to_datetime( )`.

In [ ]:
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])

commandes.dtypes      # 🔮 quel type a maintenant date_commande ?

👉 `datetime64` = un **type date**. On pourra désormais soustraire deux dates, trouver la plus récente, etc.

Lis cette ligne comme en M2.3 : Python calcule la droite (la colonne convertie en dates), puis **range le résultat dans la même colonne**, à la place de l'ancienne.

### Travailler sur une colonne
`commandes["montant"]` sélectionne **une colonne** (une Series). On écrit son nom entre crochets et guillemets, comme une clé de dictionnaire (M2.6). On peut ensuite lui appliquer un calcul :

| Calcul | Code |
|---|---|
| somme | `commandes["montant"].sum()` |
| moyenne | `.mean()` |
| plus grande / plus petite valeur | `.max()` / `.min()` |
| nombre de valeurs | `len(commandes["montant"])` ou `.count()` |

In [ ]:
print("Chiffre d'affaires total :", commandes["montant"].sum(), "€")
print("Montant moyen d'une commande :", commandes["montant"].mean(), "€")

### ✏️ Exercice 3.1 — Les chiffres clés du fichier
Calcule et affiche :
1. le **nombre de commandes** ;
2. la **plus grosse commande** (en €) ;
3. le **nombre moyen d'articles** par commande, arrondi à 1 décimale ;
4. la **date de la première** et de la **dernière commande**.

💡 *Indice :* `len(commandes)`, `.max()`, `round(... , 1)`. Pour les dates, `.min()` et `.max()` fonctionnent aussi sur `commandes["date_commande"]`.
🚀 *Bonus :* le chiffre d'affaires total arrondi à l'euro près.

In [ ]:
nb_commandes = A_COMPLETER   # ✏️ À COMPLÉTER
plus_grosse = A_COMPLETER   # ✏️ À COMPLÉTER
articles_moyen = A_COMPLETER   # ✏️ À COMPLÉTER
premiere_date = A_COMPLETER   # ✏️ À COMPLÉTER
derniere_date = A_COMPLETER   # ✏️ À COMPLÉTER

print("Nombre de commandes :", nb_commandes)
print("Plus grosse commande :", plus_grosse, "€")
print("Articles par commande :", articles_moyen)
print("Du", premiere_date, "au", derniere_date)

In [ ]:
# @title 🆘 Cellule de secours — exercice 3.1 (la solution : à n'ouvrir qu'après avoir essayé)
print("Nombre de commandes :", len(commandes))
print("Plus grosse commande :", commandes["montant"].max(), "€")
print("Articles par commande :", round(commandes["nb_articles"].mean(), 1))
print("Du", commandes["date_commande"].min(), "au", commandes["date_commande"].max())

🔴 **Erreur fréquente :** un nom de colonne mal écrit (majuscule, accent, faute de frappe). Lis la dernière ligne :

In [ ]:
commandes["Montant"]

👉 `KeyError: 'Montant'` : la colonne s'appelle `montant`, sans majuscule. En cas de doute, `commandes.columns` affiche la liste exacte des noms.

# M4 · Étape 2 — Explorer : où, quoi, et combien de fois ?
👥 **On inverse les rôles !**

**Le problème business :** avant de segmenter, on veut comprendre l'activité. Où les clients achètent-ils ? Quelles catégories rapportent le plus ? Et surtout : **les clients reviennent-ils ?**

On utilise 3 outils, du plus simple au plus puissant : **compter** (`value_counts`), **filtrer**, **regrouper** (`groupby`).

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le tableau commandes, avec date_commande convertie en date (M3)
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import matplotlib.pyplot as plt

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "database.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

### Outil 1 — Compter : `value_counts()`
`value_counts()` compte combien de fois chaque valeur apparaît dans une colonne, de la plus fréquente à la moins fréquente.

In [ ]:
commandes["canal"].value_counts()

In [ ]:
# normalize=True : en proportion plutôt qu'en nombre (0.69 = 69 %)
commandes["canal"].value_counts(normalize=True)

### Outil 2 — Filtrer : garder seulement certaines lignes
Filtrer se fait en **deux temps**. Exécute les deux cellules et compare :

**Temps 1 : poser une question à chaque ligne.** Une comparaison (M2.4) appliquée à une colonne répond `True` ou `False` **pour chaque commande**.

In [ ]:
est_en_boutique = commandes["canal"] == "boutique"
est_en_boutique.head()        # une réponse True / False par commande

**Temps 2 : garder les lignes qui ont répondu `True`.** On met la question entre crochets, derrière le tableau.

In [ ]:
commandes_boutique = commandes[est_en_boutique]

print("Commandes en boutique :", len(commandes_boutique))
commandes_boutique.head()

👉 En pratique, on écrit souvent les deux temps en une seule ligne : `commandes[commandes["canal"] == "boutique"]`. Ça se lit : « dans `commandes`, garde les lignes où le canal vaut boutique ».

### Outil 3 — Regrouper : `groupby()`
**Question :** quel chiffre d'affaires par canal ? C'est un **tableau croisé dynamique**, en une ligne :

In [ ]:
commandes.groupby("canal")["montant"].sum()

**Se lit de gauche à droite, en 3 morceaux :**
| Morceau | Ce qu'il fait | Dans Excel |
|---|---|---|
| `commandes.groupby("canal")` | fait des paquets de lignes : un paquet par canal | les lignes du tableau croisé |
| `["montant"]` | dans chaque paquet, prend la colonne montant | le champ « valeurs » |
| `.sum()` | additionne les montants de chaque paquet | « somme de » |

On peut remplacer `.sum()` par `.mean()`, `.count()`, `.max()`…

### ✏️ Exercice 4.1 — Canaux et catégories
1. Le **nombre de commandes par catégorie** ;
2. le **chiffre d'affaires par catégorie**, trié du plus grand au plus petit ;
3. le **montant moyen d'une commande par canal**, arrondi à 2 décimales.

💡 *Indice :* 1) `value_counts()` sur la colonne `categorie` · 2) `groupby("categorie")["montant"].sum()`, puis `.sort_values(ascending=False)` · 3) `groupby("canal")` puis `.mean().round(2)`.
🚀 *Bonus :* affiche le CA par catégorie sous forme de graphique en barres horizontales : ajoute `.plot(kind="barh")` à la fin de la ligne 2.

In [ ]:
print(A_COMPLETER)   # ✏️ À COMPLÉTER
print(A_COMPLETER)   # ✏️ À COMPLÉTER
print(A_COMPLETER)   # ✏️ À COMPLÉTER

In [ ]:
# @title 🆘 Cellule de secours — exercice 4.1 (la solution : à n'ouvrir qu'après avoir essayé)
print(commandes["categorie"].value_counts())
print(commandes.groupby("categorie")["montant"].sum().sort_values(ascending=False))
print(commandes.groupby("canal")["montant"].mean().round(2))

👉 **Lecture business :** le site fait plus de commandes, mais le **panier moyen est plus élevé en boutique** (conseil en magasin, achats groupés). Une info utile pour la suite.

### La question clé : les clients reviennent-ils ?
4 475 commandes… mais combien de **clients différents** ? `nunique()` compte les valeurs **uniques** d'une colonne.

In [ ]:
print("Commandes :", len(commandes))
print("Clients différents :", commandes["id_client"].nunique())

👉 **800 clients pour 4 475 commandes** : en moyenne, un client a commandé plus de 5 fois. Regardons l'historique d'un client précis, avec un filtre :

In [ ]:
commandes[commandes["id_client"] == "C0001"]

👉 Le client `C0001` a **15 lignes** : 15 commandes en deux ans, en boutique à Marseille et sur le site. C'est exactement ce qu'on attend d'un vrai fichier de ventes.

### ✏️ Exercice 4.2 — Combien de fois chaque client a-t-il commandé ?
1. Compte le **nombre de commandes par client** avec `value_counts()` sur `id_client`, et range le résultat dans `commandes_par_client`. Affiche les 5 premiers.
2. Affiche l'historique du client `C0004` : combien de commandes a-t-il passées ?
3. Combien de clients n'ont commandé **qu'une seule fois** ?

💡 *Indice :* 3) `commandes_par_client == 1` donne `True` / `False` pour chaque client ; `.sum()` compte les `True` (un `True` vaut 1).
🚀 *Bonus :* le nombre moyen de commandes par client, arrondi à 1 décimale.

In [ ]:
commandes_par_client = A_COMPLETER   # ✏️ À COMPLÉTER
print(commandes_par_client.head())

print(commandes[A_COMPLETER])   # ✏️ À COMPLÉTER

nb_une_seule_fois = A_COMPLETER   # ✏️ À COMPLÉTER
print("Clients avec une seule commande :", nb_une_seule_fois)

In [ ]:
# @title 🆘 Cellule de secours — exercice 4.2 (la solution : à n'ouvrir qu'après avoir essayé)
commandes_par_client = commandes["id_client"].value_counts()
print(commandes_par_client.head())
print(commandes[commandes["id_client"] == "C0004"])
print("Clients avec une seule commande :", (commandes_par_client == 1).sum())

### 🎯 Ce qu'on retient pour la suite
Certains clients ont commandé **22 fois**, d'autres **une seule fois**. Certains ont commandé hier, d'autres il y a un an et demi. **Ce ne sont pas les mêmes clients**, et ils ne devraient pas recevoir le même message.

Mais pour les comparer, il faut **une ligne par client**, pas une ligne par commande. C'est l'étape suivante.

# M5 · Étape 3 — Des commandes aux clients : le tableau RFM
👥 **On inverse les rôles !**

**Le problème :** le k-means va regrouper des **clients**. Il lui faut donc un tableau avec **une ligne par client**, et des **chiffres** qui décrivent chaque client.

Les équipes CRM résument un client avec 3 chiffres, la méthode **RFM** :

| Lettre | Indicateur | Question marketing | Calcul à partir des commandes |
|---|---|---|---|
| **R** | Récence | A-t-il commandé récemment ? | jours entre sa **dernière** commande et aujourd'hui |
| **F** | Fréquence | Revient-il souvent ? | **nombre** de ses commandes |
| **M** | Montant | Combien dépense-t-il ? | **somme** de ses commandes, puis panier moyen |

On construit ce tableau **en 4 étapes**.

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le tableau commandes, avec date_commande convertie en date (M3)
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import matplotlib.pyplot as plt

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "database.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

### Étape 3.1 — Fixer la date d'aujourd'hui
Pour calculer « il y a combien de jours », il faut une date de référence. Les données s'arrêtent fin octobre 2026 : on fait comme si l'analyse avait lieu le **1ᵉʳ novembre 2026**.

In [ ]:
DATE_REFERENCE = pd.Timestamp("2026-11-01")     # Timestamp = une date pour pandas
print("Date de l'analyse :", DATE_REFERENCE)

### Étape 3.2 — Un seul calcul par client : `groupby`
On reprend le `groupby` de M4, mais en regroupant par **client**. 🔮 Que va donner cette cellule ?

In [ ]:
derniere_commande = commandes.groupby("id_client")["date_commande"].max()
derniere_commande.head()

👉 Pour chaque client, la date de sa commande **la plus récente** (`max` sur des dates = la plus tardive). On passe de 4 475 lignes à **800**.

### Étape 3.3 — Plusieurs calculs d'un coup : `agg`
`.agg( )` fait **plusieurs calculs en même temps**, chacun dans une nouvelle colonne. Chaque ligne suit le même modèle :

`nom_de_la_nouvelle_colonne = ("colonne_de_départ", "calcul")`

In [ ]:
clients = commandes.groupby("id_client").agg(
    derniere_commande=("date_commande", "max"),     # R : la date de sa dernière commande
    frequence=("id_commande", "count"),             # F : combien de commandes
    montant_total=("montant", "sum"),               # M : combien il a dépensé au total
)

print(clients.shape)       # 🔮 combien de lignes ?
clients.head()

| Ligne | Se lit | Résultat pour `C0001` |
|---|---|---|
| `derniere_commande=("date_commande", "max")` | la date la plus récente parmi ses commandes | 2026-10-10 |
| `frequence=("id_commande", "count")` | le nombre de ses commandes | 15 |
| `montant_total=("montant", "sum")` | la somme de ses montants | 1254.42 € |

👉 `id_client` n'est plus une colonne : il est passé **à gauche, en gras**. C'est l'**index**, l'étiquette de chaque ligne. On s'en sert juste après.

### Étape 3.4 — Calculer la récence en jours
On soustrait deux dates : le résultat est une **durée**. `.dt.days` la convertit en **nombre de jours**.

In [ ]:
clients["recence_jours"] = (DATE_REFERENCE - clients["derniere_commande"]).dt.days

clients.head()

👉 Exactement le même principe qu'en M2.3 : Python calcule la droite pour les 800 clients d'un coup, et range le résultat dans une **nouvelle colonne**. Plus besoin de boucle `for` : pandas applique le calcul à chaque ligne.

### ✏️ Exercice 5.1 — Le panier moyen
Ajoute une colonne `panier_moyen` : le **montant moyen d'une commande** pour chaque client (montant total ÷ fréquence), arrondi à 2 décimales.

💡 *Indice :* même modèle que la récence : `clients["panier_moyen"] = (clients[...] / clients[...]).round(2)`.
🚀 *Bonus :* affiche les 5 clients qui ont dépensé le plus au total : `clients.sort_values("montant_total", ascending=False).head()`.

In [ ]:
clients["panier_moyen"] = A_COMPLETER   # ✏️ À COMPLÉTER

clients.head()

In [ ]:
# @title 🆘 Cellule de secours — exercice 5.1 (la solution : à n'ouvrir qu'après avoir essayé)
clients["panier_moyen"] = (clients["montant_total"] / clients["frequence"]).round(2)
clients.head()

### Étape de contrôle — Vérifier un client à la main
Un analyste ne fait jamais confiance à un calcul sans l'avoir vérifié sur un exemple. `.loc["C0001"]` affiche **la ligne** du client `C0001`, grâce à l'index.

In [ ]:
clients.loc["C0001"]

### ✏️ Exercice 5.2 — Contrôle qualité
À partir du tableau **`commandes`** (pas de `clients`), recalcule pour le client `C0001` :
1. son **nombre de commandes** ;
2. la **date de sa dernière commande** ;
3. son **montant total**.

Les 3 résultats doivent être identiques à la ligne affichée juste au-dessus.

💡 *Indice :* commence par filtrer ses commandes (M4) : `commandes_c1 = commandes[commandes["id_client"] == "C0001"]`. Ensuite : `len( )`, `.max()`, `.sum()` sur les bonnes colonnes.
🚀 *Bonus :* fais la même vérification pour le client `C0005`.

In [ ]:
commandes_c1 = A_COMPLETER   # ✏️ À COMPLÉTER

print("Nombre de commandes :", A_COMPLETER)   # ✏️ À COMPLÉTER
print("Dernière commande :", A_COMPLETER)   # ✏️ À COMPLÉTER
print("Montant total :", A_COMPLETER)   # ✏️ À COMPLÉTER

In [ ]:
# @title 🆘 Cellule de secours — exercice 5.2 (la solution : à n'ouvrir qu'après avoir essayé)
commandes_c1 = commandes[commandes["id_client"] == "C0001"]
print("Nombre de commandes :", len(commandes_c1))
print("Dernière commande :", commandes_c1["date_commande"].max())
print("Montant total :", round(commandes_c1["montant"].sum(), 2))

### Un coup d'œil sur les 800 clients
`.describe()` résume chaque colonne chiffrée : nombre de valeurs, moyenne, écart-type (`std`), minimum, quartiles, maximum.

In [ ]:
clients[["recence_jours", "frequence", "panier_moyen"]].describe().round(1)

**Comment le lire :**
- `recence_jours` va de **1 à 700 jours** : certains clients ont commandé hier, d'autres il y a plus d'un an et demi ;
- `frequence` va de **1 à 22** commandes ;
- `panier_moyen` va d'environ **19 € à 192 €**.

🎯 **Le tableau est prêt :** 800 lignes, une par client, et 3 chiffres pour décrire chacun. On peut passer au k-means.

# M6 · Étape 4 — Le k-means, pas à pas
👥 **On inverse les rôles !**

### Ce que fait le k-means
Le **k-means** est un algorithme de **machine learning** qui range des individus en **k groupes** (k = un nombre qu'on choisit), de façon que les individus d'un même groupe se ressemblent le plus possible. Personne ne lui dit quels groupes chercher : il les trouve seul dans les données. C'est de l'**apprentissage non supervisé**.

**Comment il s'y prend, en 4 temps :**
1. il place **k points au hasard** : les « centres » des futurs groupes ;
2. chaque client rejoint **le centre le plus proche** de lui ;
3. chaque centre se déplace **au milieu** de son groupe ;
4. on **recommence** les temps 2 et 3, jusqu'à ce que plus rien ne bouge.

Une image : on ouvre k boutiques dans une ville. Chaque habitant va à la boutique la plus proche, puis chaque boutique déménage au centre de sa clientèle, et ainsi de suite jusqu'à ce que les emplacements soient stables.

« Proche » veut dire : **des chiffres RFM qui se ressemblent**. Deux clients qui ont commandé récemment, souvent, avec des paniers proches, sont « proches ».

**Les 4 étapes dans le code :** choisir les variables → les mettre à la même échelle → choisir k → lancer l'algorithme.

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le tableau clients (RFM) construit en M5
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import matplotlib.pyplot as plt

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "database.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

DATE_REFERENCE = pd.Timestamp("2026-11-01")
clients = commandes.groupby("id_client").agg(
    derniere_commande=("date_commande", "max"),
    frequence=("id_commande", "count"),
    montant_total=("montant", "sum"),
)
clients["recence_jours"] = (DATE_REFERENCE - clients["derniere_commande"]).dt.days
clients["panier_moyen"] = (clients["montant_total"] / clients["frequence"]).round(2)
print("✅ clients est en mémoire :", len(clients), "clients")

### Étape 4.1 — Choisir les variables
On décrit chaque client avec **3 chiffres** : récence, fréquence, panier moyen.

Pourquoi pas `montant_total` ? Parce qu'il est **déjà contenu** dans les deux autres (montant total = fréquence × panier moyen). L'ajouter compterait deux fois la même information.

In [ ]:
VARIABLES = ["recence_jours", "frequence", "panier_moyen"]    # une liste (M2.5) de noms de colonnes

X = clients[VARIABLES]      # on garde seulement ces 3 colonnes : X est le tableau donné à l'algorithme
X.head()

### Étape 4.2 — Mettre les variables à la même échelle (standardiser)
**Le problème :** le k-means mesure des **distances**. Or les colonnes n'ont pas la même échelle :
- entre deux clients, la récence peut différer de **300** (jours) ;
- la fréquence, de **5** (commandes).

Sans précaution, l'algorithme ne « voit » que la récence, parce que ses écarts sont des centaines de fois plus grands. 5 commandes d'écart compteraient autant que 5 jours, c'est-à-dire presque rien.

**La solution : standardiser.** Chaque valeur est transformée en « **combien d'écarts-types au-dessus ou en dessous de la moyenne** ». Après ça, chaque colonne a une **moyenne de 0** et un **écart-type de 1** : elles pèsent toutes autant.

| Valeur standardisée | Se lit |
|---|---|
| `0` | dans la moyenne |
| `+2` | très au-dessus de la moyenne |
| `-1` | en dessous de la moyenne |

In [ ]:
from sklearn.preprocessing import StandardScaler    # scikit-learn : LA librairie de machine learning en Python

scaler = StandardScaler()            # 1. on crée l'outil
X_std = scaler.fit_transform(X)      # 2. il calcule moyenne et écart-type de chaque colonne, puis transforme

# Pour vérifier, on remet le résultat dans un tableau lisible
pd.DataFrame(X_std, columns=VARIABLES).describe().round(2)

👉 Les 3 colonnes ont maintenant `mean` = **0** et `std` = **1**. Elles sont comparables.

### Étape 4.3 — Choisir k : la méthode du coude
**Combien de groupes ?** Le k-means ne le décide pas : c'est à nous de fixer k. Pour nous aider, on mesure l'**inertie** : la distance totale entre chaque client et le centre de son groupe. Plus elle est petite, plus les groupes sont serrés.

On essaie k = 1, 2, 3… 8 avec une **boucle `for`** et un **accumulateur** (M2.8) : on range l'inertie de chaque essai dans une liste.

In [ ]:
from sklearn.cluster import KMeans

inerties = []                                          # avant la boucle : une liste vide

for k in range(1, 9):                                  # k prend les valeurs 1, 2, 3 … 8
    modele = KMeans(n_clusters=k, n_init=10, random_state=42)
    modele.fit(X_std)                                  # on lance le k-means avec k groupes
    inerties.append(modele.inertia_)                   # dans la boucle : on ajoute l'inertie à la liste

plt.plot(range(1, 9), inerties, marker="o")            # après la boucle : on trace le résultat
plt.xlabel("Nombre de groupes (k)")
plt.ylabel("Inertie")
plt.title("La méthode du coude")
plt.show()

**Comment lire ce graphique :**
- avec plus de groupes, l'inertie baisse toujours (avec 800 groupes, chaque client serait seul et l'inertie vaudrait 0) ;
- on cherche le **coude** : le point à partir duquel ajouter un groupe ne fait **presque plus baisser** l'inertie ;
- ici, la courbe chute fort jusqu'à **3**, encore un peu jusqu'à **4**, puis s'aplatit.

👉 Le graphique propose 3 ou 4. **Le métier tranche :** avec 4 groupes, on sépare les clients récents qui n'ont commandé qu'une ou deux fois (à accueillir) de ceux qui achètent par intermittence (à relancer avec une offre). Ce ne sont pas les mêmes messages.

### ✏️ Exercice 6.1 — Fixer k
Range dans la variable `K` le nombre de groupes que tu retiens.

💡 *Indice :* un nombre entier, sans guillemets.

In [ ]:
K = A_COMPLETER   # ✏️ À COMPLÉTER
print("Nombre de segments retenu :", K)

In [ ]:
# @title 🆘 Cellule de secours — exercice 6.1 (la solution : à n'ouvrir qu'après avoir essayé)
K = 4
print("Nombre de segments retenu :", K)

### Étape 4.4 — Lancer le k-means
Deux lignes suffisent :

| Code | Rôle |
|---|---|
| `KMeans(n_clusters=K, ...)` | crée l'algorithme, réglé pour chercher K groupes |
| `n_init=10` | le point de départ est tiré au hasard : on lance l'algorithme 10 fois et on garde le meilleur résultat |
| `random_state=42` | fige le hasard : **tout le monde obtient le même résultat** |
| `.fit_predict(X_std)` | **fit** = trouve les groupes · **predict** = donne le numéro de groupe de chaque client |

### ✏️ Exercice 6.2 — Segmenter les 800 clients
Complète la cellule pour :
1. créer l'algorithme avec `K` groupes ;
2. le lancer sur les données **standardisées** et ranger le numéro de segment de chaque client dans une nouvelle colonne `segment` ;
3. compter le nombre de clients par segment.

💡 *Indice :* `n_clusters=K` · `kmeans.fit_predict(X_std)` · `value_counts()` sur la colonne `segment`.
🚀 *Bonus :* relance avec `random_state=0`. Les numéros des segments peuvent changer, mais la taille des groupes reste la même : les numéros sont de simples étiquettes.

In [ ]:
kmeans = KMeans(n_clusters=A_COMPLETER, n_init=10, random_state=42)   # ✏️ À COMPLÉTER
clients["segment"] = kmeans.fit_predict(A_COMPLETER)   # ✏️ À COMPLÉTER

print(clients[A_COMPLETER].value_counts())   # ✏️ À COMPLÉTER
clients.head()

In [ ]:
# @title 🆘 Cellule de secours — exercice 6.2 (la solution : à n'ouvrir qu'après avoir essayé)
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
clients["segment"] = kmeans.fit_predict(X_std)
print(clients["segment"].value_counts())

🎉 **Chaque client a maintenant un numéro de segment, de 0 à 3.** Mais un numéro ne dit rien : « segment 2 » ne parle à personne en réunion. L'étape suivante consiste à **comprendre** chaque groupe.

### 🔬 La preuve : pourquoi on a standardisé
Voici le même k-means **sans** standardisation, sur les chiffres bruts. Compare les moyennes des groupes :

In [ ]:
test = clients[VARIABLES].copy()
test["groupe_sans_standardisation"] = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(X)

test.groupby("groupe_sans_standardisation")[VARIABLES].mean().round(1)

👉 **Sans standardisation, les groupes sont découpés presque uniquement selon la récence** (des tranches de jours), et des clients très fidèles se retrouvent mélangés avec des clients d'une seule commande. **Règle d'or : on standardise toujours avant un k-means.**

# M7 · Étape 5 — Lire les segments et passer à l'action
👥 **On inverse les rôles !**

**Le problème business :** on a 4 numéros. La directrice marketing veut savoir **qui** sont ces clients, **ce qu'ils pèsent** dans le chiffre d'affaires, et **quoi leur envoyer**.

In [ ]:
# 🔁 CHECKPOINT — recrée les données dont ce module a besoin
# Ce module réutilise : le tableau clients avec la colonne segment (M6)
# → Tout s'est bien passé jusqu'ici ? L'exécuter ne change rien, mais ça ne casse rien non plus.
# → Tu as eu une erreur, sauté une section ou redémarré la session ? Exécute-la avant de continuer.

import pandas as pd
import matplotlib.pyplot as plt

URL_DATA = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/"
commandes = pd.read_csv(URL_DATA + "database.csv")
commandes["date_commande"] = pd.to_datetime(commandes["date_commande"])
print("✅ commandes est en mémoire :", len(commandes), "commandes")

DATE_REFERENCE = pd.Timestamp("2026-11-01")
clients = commandes.groupby("id_client").agg(
    derniere_commande=("date_commande", "max"),
    frequence=("id_commande", "count"),
    montant_total=("montant", "sum"),
)
clients["recence_jours"] = (DATE_REFERENCE - clients["derniere_commande"]).dt.days
clients["panier_moyen"] = (clients["montant_total"] / clients["frequence"]).round(2)
print("✅ clients est en mémoire :", len(clients), "clients")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

VARIABLES = ["recence_jours", "frequence", "panier_moyen"]
X_std = StandardScaler().fit_transform(clients[VARIABLES])
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
clients["segment"] = kmeans.fit_predict(X_std)

def nommer_segments(df):
    """Donne un nom à chaque numéro de segment, à partir des moyennes."""
    moyennes = df.groupby("segment")[["recence_jours", "frequence"]].mean()
    fideles = moyennes["frequence"].idxmax()          # ceux qui commandent le plus souvent
    dormants = moyennes["recence_jours"].idxmax()     # ceux qui n'ont pas commandé depuis longtemps
    reste = moyennes.drop(index=[fideles, dormants])
    nouveaux = reste["recence_jours"].idxmin()        # parmi les autres : les plus récents
    occasionnels = reste.drop(index=nouveaux).index[0]
    return {fideles: "Fidèles", dormants: "Dormants", nouveaux: "Nouveaux", occasionnels: "Occasionnels"}

clients["persona"] = clients["segment"].map(nommer_segments(clients))
print("✅ Segmentation en mémoire :")
print(clients["persona"].value_counts())

### Étape 5.1 — Le portrait-robot de chaque segment
On calcule la **moyenne** de chaque variable **par segment** : c'est un `groupby`, comme en M4.

In [ ]:
profils = clients.groupby("segment")[VARIABLES].mean().round(1)
profils

### ✏️ Exercice 7.1 — Compléter le portrait
Ajoute au tableau `profils` deux colonnes :
1. `nb_clients` : le nombre de clients de chaque segment ;
2. `part_ca_%` : la part du chiffre d'affaires total réalisée par chaque segment, en %, arrondie à 1 décimale.

💡 *Indice :* 1) `clients["segment"].value_counts()` · 2) le CA par segment est `clients.groupby("segment")["montant_total"].sum()` ; divise-le par le CA total `clients["montant_total"].sum()`, puis multiplie par 100.
🚀 *Bonus :* trie `profils` par part du CA décroissante.

In [ ]:
profils["nb_clients"] = A_COMPLETER   # ✏️ À COMPLÉTER
ca_par_segment = clients.groupby("segment")["montant_total"].sum()
profils["part_ca_%"] = A_COMPLETER   # ✏️ À COMPLÉTER

profils

In [ ]:
# @title 🆘 Cellule de secours — exercice 7.1 (la solution : à n'ouvrir qu'après avoir essayé)
profils = clients.groupby("segment")[VARIABLES].mean().round(1)
profils["nb_clients"] = clients["segment"].value_counts()
ca_par_segment = clients.groupby("segment")["montant_total"].sum()
profils["part_ca_%"] = (ca_par_segment / clients["montant_total"].sum() * 100).round(1)
profils

### Étape 5.2 — Donner un nom à chaque segment
Lis le tableau `profils` ligne par ligne et réponds à 3 questions pour chaque segment :
- **Récence** : a-t-il commandé récemment (quelques jours ou semaines) ou il y a très longtemps (plus d'un an) ?
- **Fréquence** : beaucoup de commandes, ou une ou deux ?
- **Panier** : gros, moyen ou petit ?

| Si tu lis… | Le segment s'appelle |
|---|---|
| commandes récentes + très fréquentes + gros panier + grosse part du CA | **Fidèles** |
| commandes récentes + 1 ou 2 commandes seulement | **Nouveaux** |
| dernière commande il y a plusieurs mois + petit panier | **Occasionnels** |
| dernière commande il y a plus d'un an | **Dormants** |

### ✏️ Exercice 7.2 — Nommer les segments
Dans le dictionnaire `noms`, associe **ton** numéro de segment au bon nom. ⚠️ Les numéros dépendent de ton tableau `profils` : lis-le avant de remplir.

💡 *Indice :* un dictionnaire (M2.6) dont les clés sont les numéros (sans guillemets) et les valeurs les noms (entre guillemets). `.map(noms)` remplace chaque numéro par son nom.

In [ ]:
noms = {
    0: "A_COMPLETER",   # ✏️ À COMPLÉTER
    1: "A_COMPLETER",   # ✏️ À COMPLÉTER
    2: "A_COMPLETER",   # ✏️ À COMPLÉTER
    3: "A_COMPLETER",   # ✏️ À COMPLÉTER
}
clients["persona"] = clients["segment"].map(noms)

clients["persona"].value_counts()

In [ ]:
# @title 🆘 Cellule de secours — exercice 7.2 (la solution : à n'ouvrir qu'après avoir essayé)
moyennes = clients.groupby("segment")[["recence_jours", "frequence"]].mean()
fideles = moyennes["frequence"].idxmax()
dormants = moyennes["recence_jours"].idxmax()
reste = moyennes.drop(index=[fideles, dormants])
nouveaux = reste["recence_jours"].idxmin()
occasionnels = reste.drop(index=nouveaux).index[0]
noms = {fideles: "Fidèles", dormants: "Dormants", nouveaux: "Nouveaux", occasionnels: "Occasionnels"}
clients["persona"] = clients["segment"].map(noms)
print(clients["persona"].value_counts())

### Étape 5.3 — Voir les segments
Chaque point est un client. On dessine **un nuage de points par persona**, avec une boucle `for` (M2.8) : chaque tour ajoute une couleur.

In [ ]:
for persona in clients["persona"].unique():                     # un tour par persona
    groupe = clients[clients["persona"] == persona]              # filtre (M4) : les clients de ce persona
    plt.scatter(groupe["recence_jours"], groupe["frequence"], label=persona, alpha=0.6)

plt.xlabel("Récence (jours depuis la dernière commande)")
plt.ylabel("Fréquence (nombre de commandes)")
plt.title("Les 4 segments d'Amora Zone")
plt.legend()
plt.show()

👉 Les Fidèles sont **en haut à gauche** (récents et fréquents), les Dormants **à droite** (pas de commande depuis longtemps), les Nouveaux **en bas à gauche** (récents mais une ou deux commandes).

### Étape 5.4 — Revenir aux commandes pour mieux comprendre
Les commandes contiennent des infos qu'on n'a pas données au k-means : **canal**, **code promo**, **catégorie**. On ajoute le persona de chaque client à **chacune de ses commandes**, avec `merge` (l'équivalent d'une RECHERCHEV dans Excel).

In [ ]:
commandes_seg = commandes.merge(clients[["persona"]], left_on="id_client", right_index=True)
#   commandes.merge(...)            : on part du tableau des commandes
#   clients[["persona"]]            : on va chercher le persona dans le tableau clients
#   left_on="id_client"             : la clé côté commandes : la colonne id_client
#   right_index=True                : la clé côté clients : son index (id_client, M5)

commandes_seg.head()

**Le canal préféré de chaque persona.** `pd.crosstab` croise deux colonnes ; `normalize="index"` donne des pourcentages par ligne (chaque ligne fait 100 %).

In [ ]:
pd.crosstab(commandes_seg["persona"], commandes_seg["canal"], normalize="index").round(2)

### ✏️ Exercice 7.3 — Qui utilise des codes promo ?
1. Crée une colonne `a_un_code` qui vaut `True` si la commande a utilisé un code promo.
2. Calcule, **par persona**, la part des commandes avec un code promo.

💡 *Indice :* 1) `.notna()` répond `True` quand une case n'est **pas** vide · 2) la moyenne d'une colonne de `True` / `False` donne la part de `True` : `groupby("persona")["a_un_code"].mean()`.
🚀 *Bonus :* la catégorie préférée de chaque persona, avec `pd.crosstab` sur `persona` et `categorie`.

In [ ]:
commandes_seg["a_un_code"] = A_COMPLETER   # ✏️ À COMPLÉTER

print(commandes_seg.groupby(A_COMPLETER)[A_COMPLETER].mean().round(2))   # ✏️ À COMPLÉTER

In [ ]:
# @title 🆘 Cellule de secours — exercice 7.3 (la solution : à n'ouvrir qu'après avoir essayé)
commandes_seg["a_un_code"] = commandes_seg["code_promo"].notna()
print(commandes_seg.groupby("persona")["a_un_code"].mean().round(2))

👉 **Lecture business :**
- les **Occasionnels** utilisent un code promo dans environ **6 commandes sur 10**, surtout sur le site : ils sont sensibles au prix ;
- les **Nouveaux** en utilisent un dans **1 commande sur 2** (souvent le code de bienvenue) ;
- les **Fidèles** sont ceux qui achètent le plus **en boutique** (45 % de leurs commandes, contre environ 10 % pour les Nouveaux et les Occasionnels), et presque jamais avec un code : inutile de leur donner une remise, ils viendraient quand même.

### Étape 5.5 — Le plan d'action
### ✏️ Exercice 7.4 — En binôme : une action par persona
Pour chaque persona, propose **un objectif**, **une action** et **un canal** (email, SMS, en boutique, publicité…). Appuie-toi sur les chiffres : part du CA, récence, canal préféré, usage des codes promo.

🚀 *Bonus :* si tu n'as le budget que pour **une seule** campagne, quel persona choisis-tu ? Justifie avec le tableau `profils`.

In [ ]:
plan_action = {
    "Fidèles":      "A_COMPLETER",   # ✏️ À COMPLÉTER
    "Nouveaux":     "A_COMPLETER",   # ✏️ À COMPLÉTER
    "Occasionnels": "A_COMPLETER",   # ✏️ À COMPLÉTER
    "Dormants":     "A_COMPLETER",   # ✏️ À COMPLÉTER
}
for persona, action in plan_action.items():
    print(persona, "→", action)

### ⚠️ Les 5 pièges du k-means
1. **Oublier de standardiser** → les groupes ne reflètent qu'une seule variable (tu l'as vu en M6).
2. **Croire que k est imposé par les maths** → le coude propose, le métier décide.
3. **Lire les numéros comme un classement** → le segment 3 n'est pas « meilleur » que le 0 : ce sont des étiquettes.
4. **Les valeurs extrêmes** → un client B2B avec des paniers à 2 000 € peut former un groupe à lui seul. On les repère avec `describe()` avant de lancer l'algorithme.
5. **Confondre « trouver des groupes » et « expliquer »** → le k-means dit **qui** se ressemble, pas **pourquoi**. L'interprétation reste ton travail.

# [OPTIONNEL] M7+ · Ton template de segmentation réutilisable
Tout ce que tu as fait aujourd'hui, en **deux cellules**, pour un autre fichier de commandes (en stage, par exemple).

**Mode d'emploi :** modifie seulement la cellule « ⚙️ Réglages » (le fichier, les noms de colonnes, k), puis exécute la cellule « ▶️ Moteur ». Le fichier doit contenir **une ligne par commande**, avec au moins : un identifiant client, une date et un montant.

In [ ]:
# ⚙️ RÉGLAGES — la seule cellule à modifier
FICHIER = "https://raw.githubusercontent.com/SAMIMZOUGHI/MBADMB_python/main/data/database.csv"   # chemin ou lien vers ton CSV de commandes
COL_CLIENT = "id_client"           # la colonne qui identifie le client
COL_DATE = "date_commande"         # la colonne date
COL_MONTANT = "montant"            # la colonne montant
K = 4                              # nombre de segments (à choisir après avoir lu le coude)
SEPARATEUR = ","                   # "," ou ";" (fichiers Excel français)

In [ ]:
# ▶️ MOTEUR — à exécuter tel quel
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Charger et préparer
df = pd.read_csv(FICHIER, sep=SEPARATEUR)
df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")
df[COL_MONTANT] = pd.to_numeric(df[COL_MONTANT], errors="coerce")
df = df.dropna(subset=[COL_CLIENT, COL_DATE, COL_MONTANT])
reference = df[COL_DATE].max() + pd.Timedelta(days=1)

# 2. Commandes → clients (RFM)
rfm = df.groupby(COL_CLIENT).agg(
    derniere_commande=(COL_DATE, "max"),
    frequence=(COL_DATE, "count"),
    montant_total=(COL_MONTANT, "sum"),
)
rfm["recence_jours"] = (reference - rfm["derniere_commande"]).dt.days
rfm["panier_moyen"] = rfm["montant_total"] / rfm["frequence"]
VARS = ["recence_jours", "frequence", "panier_moyen"]
print(f"{len(df)} commandes → {len(rfm)} clients")

# 3. Standardiser, coude, k-means
X_std = StandardScaler().fit_transform(rfm[VARS])
inerties = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_std).inertia_ for k in range(1, 9)]
rfm["segment"] = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(X_std)

# 4. Graphiques
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, 9), inerties, marker="o"); axes[0].axvline(K, color="red", linestyle="--")
axes[0].set_title("Coude (ligne rouge = K choisi)"); axes[0].set_xlabel("k")
for seg in sorted(rfm["segment"].unique()):
    g = rfm[rfm["segment"] == seg]
    axes[1].scatter(g["recence_jours"], g["frequence"], label=f"Segment {seg}", alpha=0.6)
axes[1].set_xlabel("Récence (jours)"); axes[1].set_ylabel("Fréquence"); axes[1].legend()
plt.tight_layout(); plt.show()

# 5. Profils
profils_t = rfm.groupby("segment")[VARS + ["montant_total"]].mean().round(1)
profils_t.insert(0, "nb_clients", rfm["segment"].value_counts().sort_index())
profils_t["part_ca_%"] = (rfm.groupby("segment")["montant_total"].sum() / rfm["montant_total"].sum() * 100).round(1)
display(profils_t)

# 6. Export
rfm.to_csv("resultat_segmentation.csv")
print("💾 Fichier 'resultat_segmentation.csv' créé (dossier 📁 à gauche) : à télécharger.")

# [OPTIONNEL] 🎤 Démo formateur · Python demande à une IA générative de rédiger les campagnes
*Démo faite par le formateur avec sa propre clé : ne lance pas cette cellule.*
La clé est stockée dans les **Secrets de Colab** (icône 🔑 à gauche), jamais écrite dans le notebook.

Python envoie à l'IA le tableau des profils calculé en M7, et lui demande un message par segment.

In [ ]:
import requests
from google.colab import userdata

profils_txt = clients.groupby("persona")[["recence_jours", "frequence", "panier_moyen", "montant_total"]].mean().round(1).to_string()

reponse = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={
        "x-api-key": userdata.get("ANTHROPIC_API_KEY"),
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    },
    json={
        "model": "claude-sonnet-5-5",
        "max_tokens": 800,
        "messages": [{"role": "user", "content":
            "Amora Zone vend des cosmétiques naturels en boutique et en ligne. Voici les profils moyens "
            "de 4 segments clients (récence en jours, fréquence en nombre de commandes, panier et montant en €) :\n"
            + profils_txt +
            "\n\nPour chaque segment : 1) décris-le en une phrase, 2) rédige l'objet et les 2 premières lignes "
            "d'un email adapté. Ton chaleureux, tutoiement, pas de genre grammatical marqué."}],
    },
    timeout=60,
)
print(reponse.json()["content"][0]["text"])

# 🏁 Ce que tu as fait aujourd'hui

| Étape | Ce que tu as fait | En Python |
|---|---|---|
| 1 | Chargé 4 475 commandes et vérifié leurs types | `pd.read_csv`, `.head()`, `.dtypes`, `pd.to_datetime` |
| 2 | Exploré : canaux, catégories, clients qui reviennent | `value_counts`, filtres, `groupby` |
| 3 | Résumé chaque client en 3 chiffres (RFM) | `groupby().agg()`, `.dt.days`, `.loc` |
| 4 | Segmenté 800 clients avec du machine learning | `StandardScaler`, coude, `KMeans` |
| 5 | Nommé les segments et construit un plan d'action | profils, `map`, `merge`, `crosstab` |

### 📚 Pour aller plus loin
- **Cours gratuits :** `microsoft/Data-Science-For-Beginners`, `microsoft/ML-For-Beginners`, `jakevdp/PythonDataScienceHandbook`
- **S'entraîner sur pandas :** `guipsamora/pandas_exercises`
- **La documentation du k-means :** scikit-learn, page « KMeans »